In [ ]:
import os
from os import path
import pickle
import numpy as np
import shutil
from tqdm import tqdm
from scipy.stats import rankdata
from scipy.spatial.distance import squareform, cdist, pdist

from macaquethings.plotting.default_styles import *
from macaquethings.data_util.load_data import get_channel_masks
from macaquethings.plotting.anatomical import plot_data_on_anatomy
from macaquethings.rdm_util import get_rdm_design_sort_indices

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import BoundaryNorm, ListedColormap

figure_style()  # set consistent plotting defaults for all figs

In [ ]:
rdm_folder = "monkeyF_mua_minithings"
rdm_file = "monkeyF-labels_filenames-sessions_0_1_2_3_4_5-rois_3-arrays_1_2_3_4_5_6_7_8_9_10_11_12_13_14_15_16-baseline_0-standardize_1-metric_correlation-neural_mua.pkl"
# rdm_file = "monkeyF-labels_filenames-sessions_0_1_2_3_4_5-rois_1_2_3-arrays_13-baseline_0-standardize_1-metric_correlation-neural_lfp.pkl"
# rdm_file = "monkeyF-labels_filenames-sessions_0_1_2_3_4_5-rois_1_2_3-arrays_9-baseline_0-standardize_1-metric_correlation-neural_mua.pkl"
# rdm_file = "monkeyN-labels_filenames-sessions_0_3_4_5-rois_1_2_3-arrays_16-baseline_0-standardize_1-metric_correlation-neural_lfp.pkl"
# rdm_file = "monkeyF-labels_filenames-sessions_0_1_2_3_4_5-rois_1_2_3-arrays_9-baseline_0-standardize_1-metric_correlation-neural_lfp.pkl"
# rdm_file = "monkeyF-labels_filenames-sessions_0_1_2_3_4_5-rois_3-arrays_1_2_3_4_5_6_7_8_9_10_11_12_13_14_15_16-baseline_0-standardize_1-metric_correlation-neural_mua.pkl"
t_select = np.arange(0, 160, 10)
savedir = 'rdm_trajectory_panels'

rdmpath = path.join("..", "..", "results", "rdm", rdm_folder, rdm_file)

with open(rdmpath, 'rb') as f:
    rdm_data = pickle.load(f)

os.makedirs(savedir, exist_ok=True)

# Show Timecourse

In [ ]:
# get sort indices

sort_idx = get_rdm_design_sort_indices('../..', return_values=False, reduce_to_column=rdm_data['data_cfg']['labels'])

time = rdm_data["time"]
rdms = rdm_data["rdms"]

n_panels = len(t_select)

full_width = FULL_PANEL_SIZE[0]
panel_size = full_width / n_panels


plt.figure(figsize=(full_width, panel_size))
for i in range(n_panels):
    t_panel = t_select[i]
    idx = np.where(time == t_panel)[0][0]
    rdm = rdms[idx]
    rdm = rankdata(rdm)
    rdm = squareform(rdm)
    rdm = rdm[sort_idx][:, sort_idx]   
    plt.subplot(1, n_panels, i+1)
    plt.imshow(rdm, rasterized=True)
    plt.gca().axis('off')
    plt.title(f"{t_panel}ms")
plt.savefig(path.join(savedir, rdm_file[:-4] + ".svg"), dpi=800)
    

# Select a Single Time Point and Plot RDM

Generate a single RDM (e.g. for insets)

In [ ]:
# ----------------------
t_select = 130
ranktransform = True 
# ----------------------

if ranktransform:
    vmin, vmax =None, None
else:
    vmin, vmax = 0, 2

idx = np.where(time == t_select)[0][0]
rdm = rdms[idx]
if ranktransform:
    rdm = rankdata(rdm)
rdm = squareform(rdm)
rdm = rdm[sort_idx][:, sort_idx]   

plt.figure(figsize=QUARTER_PANEL_SIZE)
plt.imshow(rdm, vmin=vmin, vmax=vmax)
plt.gca().axis('off')
if not ranktransform:
    plt.colorbar(shrink=.8)
plt.title(f"{t_select} ms")
plt.savefig(path.join(savedir, rdm_file[:-4] + f"_single-t-{t_select}.svg"), dpi=800)

plt.show()

# RDM second order: time - time generalization

In [ ]:
from scipy.stats import spearmanr

spearman_rs, _ = spearmanr(rdms, axis=1)
time_time_rdm = 1 - spearman_rs

In [ ]:
plt.figure(figsize=HALF_PANEL_SIZE)
plt.imshow(time_time_rdm, rasterized=True)

xtick_idx = np.arange(5, len(time), 20)
xtick_labels = time[xtick_idx]
plt.xticks(ticks=xtick_idx, labels=xtick_labels, rotation=90)
plt.yticks(ticks=xtick_idx, labels=xtick_labels)
plt.xlabel('time (ms)')
plt.ylabel('time (ms)')
plt.colorbar(label='rank order distance (spearman)')
plt.tight_layout()
plt.savefig(path.join(savedir, rdm_file[:-4] + "_time_time_spearman.svg"), dpi=300)


# Correlate with DNN embeddings

In [ ]:
dnn_corr_savedir = "rdm_dnn_corr_panels_resnet18"
os.makedirs(dnn_corr_savedir, exist_ok=True)

model_rdm_folder = path.join("..", "..", "datasets", "TIMM", "resnet18")
# model_rdm_name = "rdms-alexnet-metric_cosine-normalization_None-pca_1000.pkl"
model_rdm_name = "rdms-resnet18-metric_cosine-normalization_None-pca_1000.pkl"

with open(path.join(model_rdm_folder, model_rdm_name), "rb") as f:
    model_rdm_data = pickle.load(f)

model_keys = model_rdm_data["selected_nodes"]
model_rdm_dict = model_rdm_data["rdms"]


def correlate_rdm_movie_with_models(rdm_timecourse, target_rdms, model_keys):
    print(f"time course shape: {rdm_timecourse.shape}")
    # extract model rdms from dict
    models = np.array([target_rdms[key] for key in model_keys])
    print(f"models shape: {models.shape}")
    return 1 - cdist(rdm_timecourse, models, metric="correlation")

# show all models
fig, axs = plt.subplots(5, 5, figsize=(full_width, full_width))
axs_flt = axs.flatten()
for i, key in enumerate(model_keys):
    model = model_rdm_dict[key]
    model = rankdata(model)
    model = squareform(model)
    model = model[sort_idx][:, sort_idx]
    ax = axs_flt[i]
    ax.imshow(model, rasterized=True)
    ax.set_title(key)
for ax in axs_flt:
    ax.axis('off')
plt.savefig(path.join(dnn_corr_savedir, 'dnn_model_rdms.svg'), dpi=500)

In [ ]:
# correlate
rdm_corrs = correlate_rdm_movie_with_models(
            rdms, model_rdm_dict, model_keys
)

In [ ]:
PANEL_SIZE = THIRD_PANEL_SIZE
WITHCOLORBAR = True 


colors = np.array(
    sns.color_palette("rocket", len(model_keys) + 1)[1:]
)  # exclude first color, since it is too faint
# sort colors by  layer position
layer_idx = np.argsort(model_rdm_data["node_indices"])
colors = colors[layer_idx]

# cmap for colorbar
cmap = ListedColormap(colors)
bounds = np.arange(len(colors) + 1)
norm = BoundaryNorm(bounds, cmap.N)

fig_rdm_corrs = plt.figure(figsize=(THIRD_WIDTH, THIRD_WIDTH / 2))

for corrs, color, name, idx in zip(
    rdm_corrs.T, colors, model_keys, model_rdm_data["node_indices"]
):
    plt.plot(time, corrs, label=f"{idx}: {name}", color=color)
plt.xlabel("time (ms)")
plt.ylabel("pearson correlation")

if WITHCOLORBAR:
    cb = plt.colorbar(
        plt.cm.ScalarMappable(norm=norm, cmap=cmap),
        ax=plt.gca(),
    )
    cb.ax.minorticks_off()
    cb.set_ticks(np.arange(len(model_rdm_data["node_indices"])) + 0.5)  # center each tick
    cb.set_ticklabels(model_keys)
    cb.ax.tick_params(labelsize=5)
    
plt.savefig(path.join(dnn_corr_savedir, f"{rdm_file[:-4]}_dnn_corrs.svg"))

# if WITHCOLORBAR is set to False, plot colorbar in separate figure
if not WITHCOLORBAR:
    fig, ax = plt.subplots(1,1, figsize=PANEL_SIZE)
    cb = plt.colorbar(
        plt.cm.ScalarMappable(norm=norm, cmap=cmap),
        ax=ax,
        shrink=.8
    )
    cb.ax.minorticks_off()
    cb.set_ticks(np.arange(len(model_rdm_data["node_indices"])) + 0.5)  # center each tick
    cb.set_ticklabels(model_keys)
    cb.ax.tick_params(labelsize=5)
    ax.axis('off')
    plt.savefig(path.join(dnn_corr_savedir, f"{rdm_file[:-4]}_dnn_corrs_cbar.svg"))
    
 

# How similar are the dnn layer 'model' RDMs?

In [ ]:
from scipy.stats import spearmanr

allmodels = np.array([model_rdm_dict[key] for key in model_keys])

rank_corrs, _ = spearmanr(allmodels, axis=1)
print(rank_corrs.shape)

plt.figure(figsize=(HALF_PANEL_SIZE[0], HALF_PANEL_SIZE[0]))
plt.imshow(rank_corrs)
plt.colorbar()
plt.xticks(np.arange(len(rank_corrs)), model_keys, rotation=90)
plt.yticks(np.arange(len(rank_corrs)), model_keys)
plt.tight_layout()
plt.savefig(path.join(dnn_corr_savedir, f"model_layer_rdm_spearman_corrs.svg"))

# Unique explained variance by each selected layer

In [ ]:
from ephyslib.rdm import unique_variance_per_model
from tqdm import tqdm
import sys
from joblib import Parallel, delayed

print(model_rdm_dict.keys())

keys_select = [
    'layer1.1.conv2',
    'layer3.1.conv2',
]

In [ ]:
models = np.array([model_rdm_dict[key] for key in keys_select])
print(models.shape)

full_vars_exp = []
partial_vars_exp = []
unique_vars_exp = []

with Parallel(return_as='generator', n_jobs=-1, verbose=100) as pool:
    res_gen = pool(delayed(unique_variance_per_model)(models, rdm, True, False, True) for rdm in rdms)
    
    for res in res_gen:
        f, p, u = res
        full_vars_exp.append(f)
        partial_vars_exp.append(p)
        unique_vars_exp.append(u)

In [ ]:
unique_vars_exp = np.array(unique_vars_exp)
full_vars_exp = np.array(full_vars_exp)
print(unique_vars_exp.shape, full_vars_exp.shape)

# indices of best layers in terms of unique explained variance
peak_unique_exp = np.max(unique_vars_exp, axis=0)
unique_exp_ranks = rankdata(peak_unique_exp)
print(unique_exp_ranks)

plt.figure(figsize=(THIRD_WIDTH, THIRD_WIDTH / 2))
for key, unique_exp, rank in zip(keys_select, unique_vars_exp.T, unique_exp_ranks):
    
    plt.plot(time, unique_exp, label=key)
plt.plot(time, full_vars_exp, color='k', label='all included models')
plt.xlabel('time (ms)')
plt.ylabel('unique explained variance')
plt.legend()
plt.savefig(path.join(dnn_corr_savedir, f"{rdm_file[:-4]}_dnn_unique_expvar_selected_layers.svg"))

# plt.figure(figsize=(HALF_PANEL_SIZE[0], 2))
# plt.xlabel('time (ms)')
# plt.ylabel('total explained variance')
# plt.tight_layout()

In [ ]:
dnn_corr_savedir